# 第6回 演習: 関数と、関数を入れた辞書

関数を書くところから始めて、関数を辞書に入れて使う。
課題1 の遅い関数は、課題3 で「答えを辞書に覚える」と一気に速くなる。
課題4 では、課題2 の電卓を、文字列で書いた命令から呼べるようにする。

## このノートブックの使い方

- セルは**上から順に**実行する (`Shift + Enter`)。途中から始めると「名前が見つからない」(`NameError`) になる
- 自分で書くのは `TODO` の行だけ。`...` や `pass` の所を、正しいコードに置き換える
- `# [CHECK]` で始まるセルは確認用。**書き換えない**
  - `OK` と出れば正解
  - `NG` と出たら、その下の行に「何がおかしいか」が書いてある
- 「名前が見つからない」と出たら、`ランタイム → 前のセルを実行` を押してから、もう一度実行する

## 全体の地図

| 課題 | 書くもの | 学ぶこと | 配点 |
|---|---|---|---|
| 課題0 | `greet`, `apply_twice` | 引数、デフォルト値、関数を引数に渡す | 10 |
| 課題1 | `tarai` | 再帰、呼び出し回数を数える | 10 |
| 課題1-b | `distinct_args` | なぜ遅いのか (同じ引数の繰り返し) | 10 |
| 課題2 | `OPS`, `calc` | 辞書に関数を入れる | 15 |
| 課題3 | `tarai_memo` | 計算した答えを辞書に覚える | 20 |
| 課題4 | `COMMANDS` | 命令の名前と関数を表に登録する | 10 |
| 課題4-b | `run` | 文字列から関数を実行する | 15 |
| 発展1・2 | `tarai_t`, `help_text` | 余力があれば | 各5 |

## 進め方

- 各課題は「説明を読む → 例題セルを実行 → `TODO` を埋める → 確認セルを実行」
- 例題セルは動きを見るためのもの。書き換えずに実行すればよい
- 終わったら出力を残したまま保存し、kibaco から提出する

In [ ]:
#@title 自動セットアップ (実行するだけ。中身は隠しています)
# [SKIP] 採点の対象外
# ---- 自己確認の準備 (check / score)。1回実行すればよい ----
import pathlib
import urllib.request

if not pathlib.Path('autograde.py').exists():
    try:
        urllib.request.urlretrieve(
            'https://3tty0n.github.io/2026-scripting-lab/grading/autograde.py', 'autograde.py')
    except Exception as e:
        print('取得できませんでした:', e)

from autograde import check, score
TASK = '06'


---
## 課題0: 関数を書く・渡す (10点)

**やること**: 2つの関数を書く。

- `greet(name, suffix='!')`: あいさつの文字列を返す。`suffix` (最後に付ける記号) は省略でき、省略すると `'!'` になる
- `apply_twice(func, x)`: 受け取った関数 `func` を、`x` に2回続けて使った結果を返す

**入出力の例**

| 呼び方 | 結果 |
|---|---|
| `greet('Alice')` | `'Hello, Alice!'` |
| `greet('Bob', '.')` | `'Hello, Bob.'` |
| `apply_twice(abs, -3)` | `3` (`abs(abs(-3))`) |
| `apply_twice(str.upper, 'ab')` | `'AB'` |

**手順**

1. `greet`: `'Hello, ' + name + suffix` を `return` する
2. `apply_twice`: `func(func(x))` を `return` する

**ヒント**: 関数も数や文字列と同じく「値」なので、引数で受け取れる。
受け取った関数は `func` という名前の引数に入っていて、`func(x)` で呼べる。

**つまずきやすい点**

- `print(...)` だけで `return` を書いていない → 結果が `None` になり NG になる
- `apply_twice` で `func(x)` を1回しか呼んでいない → `apply_twice(lambda x: x + 1, 0)` が 2 にならない

### 例題

次のセルを実行して、動きを見る。

`h=1` のように書くと、省略したときの値 (デフォルト値) になる。
`call(len, 'abc')` は、関数 `len` を引数として渡している。

In [ ]:
# 例題
def area(w, h=1):
    return w * h


print(area(3, 4), area(3))               # 12 3


def call(f, x):
    return f(x)


print(call(len, 'abc'), call(abs, -5))   # 3 5

In [ ]:
def greet(name, suffix='!'):
    return 'Hello, ' + name + suffix


def apply_twice(func, x):
    return func(func(x))


print(greet('Alice'), greet('Bob', '.'))
print(apply_twice(abs, -3), apply_twice(str.upper, 'ab'))

In [ ]:
# [CHECK] q0
need('greet', arity=2)
need('apply_twice', arity=2)

assert greet('Alice') == 'Hello, Alice!'
assert greet('Bob', '.') == 'Hello, Bob.'
assert greet(name='Eve', suffix='?') == 'Hello, Eve?', '名前付きの引数でも呼べること'
assert apply_twice(lambda x: x + 1, 0) == 2
assert apply_twice(lambda x: x * 3, 2) == 18
assert apply_twice(str.upper, 'ab') == 'AB'
print('OK')

---
## 課題1: 竹内関数 (10点)

**やること**: 竹内関数 `tarai(x, y, z)` を書き、**何回呼ばれたか**を変数 `CALLS` で数える。

竹内関数 (たらいまわし関数) は、答えは単純なのに、**呼び出し回数がとても多くなる**ことで知られる関数。
自分の中で自分自身を呼ぶ。これを**再帰** (関数が自分自身を呼ぶこと) という。

```
tarai(x, y, z) = y                                (x <= y のとき)
               = tarai(tarai(x-1, y, z),
                       tarai(y-1, z, x),
                       tarai(z-1, x, y))          (それ以外)
```

**入出力の例**

| 呼び方 | 結果 |
|---|---|
| `tarai(1, 2, 3)` | `2` (`1 <= 2` なので `y`) |
| `tarai(10, 5, 0)` | `10`。呼ばれる回数は 343,073 回 |
| `tarai(12, 6, 0)` | `12`。呼ばれる回数は 12,604,861 回 (数秒かかる) |

**手順** (呼ばれた回数を数える2行は、もう書いてある)

1. `x <= y` のときは `y` を返す
2. それ以外は、上の式の「それ以外」の行をそのまま書いて返す

**ヒント**: 関数の中から外の変数 `CALLS` を書き換えるには、`global CALLS` の1行が要る (もう書いてある)。

**つまずきやすい点**

- `呼び出し回数が ... 回` と出る → 式の `x-1` や引数の順番 (`y, z, x` など) が違う
- 実行が終わらない → 止まる条件 `x <= y` で `return y` していない

### 例題

次のセルを実行して、動きを見る。

同じ形の、もっと簡単な再帰 (フィボナッチ数)。回数の数え方はこれと同じ。

In [ ]:
# 例題
FIB_CALLS = 0


def fib(n):
    global FIB_CALLS
    FIB_CALLS += 1
    if n <= 1:
        return n
    return fib(n - 1) + fib(n - 2)


FIB_CALLS = 0
print(fib(10), FIB_CALLS)    # 55 177

`tarai` も形は同じ。止まる条件が `x <= y` で、
そうでなければ `tarai` を3回呼んだ結果を、もう一度 `tarai` に渡す。

In [ ]:
CALLS = 0


def tarai(x, y, z):
    global CALLS
    CALLS += 1
    if x <= y:
        return y
    return tarai(tarai(x-1, y, z), tarai(y-1, z, x), tarai(z-1, x, y))


import time
CALLS = 0
t0 = time.time()
print('tarai(12, 6, 0) =', tarai(12, 6, 0))
print(f'呼び出し回数: {CALLS:,} 回 / {time.time() - t0:.2f} 秒')

In [ ]:
# [CHECK] q1
need('tarai', arity=3)
need('CALLS', kind=int)

CALLS = 0; assert tarai(1, 2, 3) == 2      # x <= y なので y
CALLS = 0; assert tarai(3, 2, 1) == 3
# 自己申告の CALLS だけに頼らず、外から呼び出しそのものを観察する
CALLS = 0
with watch('tarai') as spy:
    assert tarai(10, 5, 0) == 10
assert CALLS == 343073, f'呼び出し回数が {CALLS} 回。数え方を見直すこと'
assert spy.calls == CALLS, f'外から数えると {spy.calls:,} 回。数え方がずれている'
note(f'tarai(10, 5, 0) の呼び出し {spy.calls:,} 回')

print('OK')


### 課題1-b: なぜ増えるのか (10点)

**やること**: `distinct_args(x, y, z)` を書く。`tarai(x, y, z)` を計算する途中で現れる
**引数の組 `(x, y, z)` が何種類あるか**を返す。

`tarai(12, 6, 0)` は 1,260万回も呼ばれる。しかし引数は3つとも小さい整数なので、
**同じ引数の組が何度も現れている**。それを確かめる。

**入出力の例**

| 呼び方 | 結果 |
|---|---|
| `distinct_args(1, 2, 3)` | `1` (1回で終わる) |
| `distinct_args(10, 5, 0)` | `157` |
| `distinct_args(12, 6, 0)` | `225` |

**手順** (`walk` の中に書く)

1. `seen.add((x, y, z))` で、今の引数の組を記録する
2. `x <= y` なら `y` を返す
3. それ以外は `tarai` と同じ式を返す。ただし、式の中で呼ぶのは `tarai` ではなく `walk`

**ヒント**: `seen` は**集合** (同じものを1つにまとめて持つ入れ物)。
同じ組を何回 `add` しても1つと数えるので、最後の `len(seen)` が「何種類か」になる。
`(x, y, z)` のように丸かっこでまとめたものを**タプル**という。

**つまずきやすい点**

- 確認セルに `157` 以外の数が出る → 式の中で `walk` ではなく `tarai` を呼んでいる
- `seen.add(x, y, z)` と書くとエラー → かっこを二重にして `seen.add((x, y, z))`

**注意**: `tarai(14, 7, 0)` は5億回を超える。試さないこと。

### 例題

次のセルを実行して、動きを見る。

`fib` について、同じことをやった例。

In [ ]:
# 例題
def distinct_fib(n):
    seen = set()

    def walk(n):
        seen.add(n)
        if n <= 1:
            return n
        return walk(n - 1) + walk(n - 2)

    walk(n)
    return len(seen)


print(distinct_fib(10))    # 11 (177 回呼ばれても、引数は 11 通り)

`tarai` は引数が3つなので、記録は `seen.add((x, y, z))` とタプルにする。
`walk` の中身は課題1 の `tarai` と同じ計算。ただし呼ぶ相手は `walk` 自身。

In [ ]:
def distinct_args(x, y, z):
    seen = set()

    def walk(x, y, z):
        seen.add((x, y, z))
        if x <= y:
            return y
        return walk(walk(x-1, y, z), walk(y-1, z, x), walk(z-1, x, y))

    walk(x, y, z)
    return len(seen)


print('tarai(10, 5, 0):', f'{343073:,} 回の呼び出し /',
      distinct_args(10, 5, 0), '通りの引数')
print('tarai(12, 6, 0):', f'{12604861:,} 回の呼び出し /',
      distinct_args(12, 6, 0), '通りの引数')

In [ ]:
# [CHECK] q1b
need('distinct_args', arity=3)

assert distinct_args(1, 2, 3) == 1        # 1回で終わる
assert distinct_args(10, 5, 0) == 157, distinct_args(10, 5, 0)
assert distinct_args(12, 6, 0) == 225, distinct_args(12, 6, 0)
print('OK  1,260万回の呼び出しに対して、引数の組はたった 225 通り')


---
## 課題2: 辞書に関数を入れて電卓 (15点)

**やること**: `if` を並べずに、**辞書に関数を入れて**電卓を作る。

- `OPS`: 演算子の文字列 (`'+'` など) から「2つの数を受け取る関数」を引く辞書
- `calc(op, a, b)`: `OPS` から関数を引いて、`a` と `b` を渡して呼ぶ
- 知らない演算子なら `ValueError` (値がおかしいときのエラー) を出す

**入出力の例**

| 呼び方 | 結果 |
|---|---|
| `calc('+', 3, 4)` | `7` |
| `calc('/', 20, 6)` | `3` (割り算は小数を切り捨てる) |
| `calc('**', 2, 10)` | `1024` (2 の 10 乗) |
| `calc('?', 1, 2)` | `ValueError` |

**手順**

1. `OPS` に `'-'`, `'*'`, `'/'`, `'%'`, `'**'` を、`'+'` と同じ形で足す
2. `calc`: `op` が `OPS` に無ければ `raise ValueError(op)`
3. `calc`: `OPS[op](a, b)` を返す

**ヒント**: `operator` には、演算子と同じ働きの関数がある。
`'-'` は `operator.sub`、`'*'` は `operator.mul`、`'/'` は `operator.floordiv`、
`'%'` は `operator.mod`、`'**'` は `operator.pow`。

**つまずきやすい点**

- `'/'` を `operator.truediv` にすると、`20 / 6` が `3.33...` になり NG。`floordiv` を使う
- `calc` を `if op == '+': ...` と並べて書くと NG (「表から引いて呼ぶこと」と出る)
- `OPS['+']` で取り出したものは関数。`OPS[op](a, b)` のように、後ろに `(a, b)` を付けて呼ぶ

### 例題

次のセルを実行して、動きを見る。

辞書に関数を入れて、名前で引いて呼ぶ形。

In [ ]:
# 例題
import math

FUNCS = {
    'sqrt': math.sqrt,
    'abs': abs,
}


def apply1(name, x):
    if name not in FUNCS:
        raise ValueError(f'知らない関数: {name}')
    return FUNCS[name](x)


print(apply1('sqrt', 16), apply1('abs', -3))   # 4.0 3

`calc` はこれの、引数が2つの版。`operator.add(3, 4)` は `3 + 4` と同じ。

In [ ]:
import operator

OPS = {
    '+': operator.add,
    '-': operator.sub, '*': operator.mul, '/': operator.floordiv, '%': operator.mod, '**': operator.pow,
}


def calc(op, a, b):
    if op not in OPS:
        raise ValueError(op)
    return OPS[op](a, b)


print(calc('+', 3, 4), calc('*', 3, 4), calc('**', 2, 10))

In [ ]:
# [CHECK] q2
need('OPS', kind=dict)
need('calc', arity=3)

assert calc('+', 3, 4) == 7
assert calc('-', 10, 3) == 7
assert calc('*', 3, 4) == 12
assert calc('/', 20, 6) == 3        # 整数除算にすること
assert calc('%', 20, 6) == 2
assert calc('**', 2, 10) == 1024
try:
    calc('?', 1, 2)
except ValueError:
    pass
else:
    raise AssertionError('知らない演算子は ValueError を出すこと')

# 実行中に演算子を増やせること (動的言語らしさ)
OPS['max'] = max
assert calc('max', 3, 9) == 9

# 具体例だけでなく、成り立つべき性質でも確かめる
holds('a + b は b + a と同じ', lambda a, b: calc('+', a, b) == calc('+', b, a),
      ints(2))
holds('a - b は -(b - a) と同じ', lambda a, b: calc('-', a, b) == -calc('-', b, a),
      ints(2))
holds('a * 1 は a のまま', lambda a: calc('*', a, 1) == a, ints(1))
# 書き方も見る。OPS を作っただけで if / elif に戻っていないこと
_t = tags('calc')
assert not _t or ({'subscript', 'calls:get'} & _t), \
    'OPS を作っても calc が if / elif で分けている。表から引いて呼ぶこと'

print('OK')


---
## 課題3: 答えを辞書に覚える (20点)

**やること**: `tarai` と同じ答えを返す `tarai_memo(x, y, z)` を書く。
ただし、一度計算した答えは辞書 `TARAI_CACHE` に覚えておき、同じ引数がまた来たら計算せずにそれを返す。

課題1-b で見たとおり、`tarai(12, 6, 0)` に出てくる引数の組は 225 通りしかない。
覚えておけば、計算は 225 回で済む。

- 辞書の鍵は引数の組 `(x, y, z)`、値はその答え

**入出力の例**

| 呼び方 | 結果 |
|---|---|
| `tarai_memo(12, 6, 0)` | `12`。計算した回数 `MEMO_CALLS` は 225 回 (課題1 では 1,260万回) |
| `tarai_memo(100, 50, 0)` | `100`。一瞬で終わる |

**手順** (`TODO` が3つある)

1. 覚えていたら (`key in TARAI_CACHE`)、`TARAI_CACHE[key]` を返す
2. 覚えていなければ、課題1 と同じ式で計算する。ただし、式の中で呼ぶのは `tarai_memo`
3. 計算した答え `result` を、`TARAI_CACHE[key]` に入れる

**ヒント**: タプル `(x, y, z)` は辞書の鍵にできる。下の例題 `fib_fast` と同じ形。

**つまずきやすい点**

- `... 回。覚えた答えが使われていない` と出る → 式の中で `tarai` を呼んでいる。`tarai_memo` にする
- `覚えた組が ... 通り` と出る → 手順3 で `TARAI_CACHE[key] = result` になっていない

### 例題

次のセルを実行して、動きを見る。

`fib` で同じことをした例。

In [ ]:
# 例題
FIB_CACHE = {}


def fib_fast(n):
    if n in FIB_CACHE:
        return FIB_CACHE[n]
    if n <= 1:
        result = n
    else:
        result = fib_fast(n - 1) + fib_fast(n - 2)
    FIB_CACHE[n] = result
    return result


print(fib_fast(100))    # 一瞬で返る

`tarai` は引数が3つなので、鍵を `(x, y, z)` にする。
式の中で呼ぶ相手も `tarai_memo` にすること (`tarai` を呼ぶと、覚えた答えが使われない)。

In [ ]:
TARAI_CACHE = {}
MEMO_CALLS = 0


def tarai_memo(x, y, z):
    global MEMO_CALLS
    key = (x, y, z)
    if key in TARAI_CACHE:
        return TARAI_CACHE[key]
    MEMO_CALLS += 1
    if x <= y:
        result = y
    else:
        result = tarai_memo(tarai_memo(x-1, y, z), tarai_memo(y-1, z, x), tarai_memo(z-1, x, y))
    TARAI_CACHE[key] = result
    return result


import time
MEMO_CALLS = 0
t0 = time.time()
print('tarai_memo(12, 6, 0) =', tarai_memo(12, 6, 0))
print(f'計算した回数: {MEMO_CALLS:,} 回 / {time.time() - t0:.4f} 秒')

In [ ]:
# [CHECK] q3
need('TARAI_CACHE', kind=dict)
need('tarai_memo', arity=3)

TARAI_CACHE.clear()
MEMO_CALLS = 0
assert tarai_memo(12, 6, 0) == 12
assert MEMO_CALLS == 225, f'{MEMO_CALLS} 回。覚えた答えが使われていない (課題1 では 12,604,861 回)'
assert MEMO_CALLS == distinct_args(12, 6, 0), '課題1-b の答えと一致するはず'
assert len(TARAI_CACHE) == 225, \
    f'覚えた組が {len(TARAI_CACHE)} 通り。引数の組は 225 通りのはず'
assert TARAI_CACHE[(12, 6, 0)] == 12, '鍵は (x, y, z)、値は答え'

# 答えを覚えれば、課題1 のままでは終わらない大きさも一瞬で解ける
TARAI_CACHE.clear()
assert tarai_memo(100, 50, 0) == 100
note(f'tarai_memo(100, 50, 0) は {MEMO_CALLS:,} 回の計算で終わった')

print('OK  1,260万回の呼び出しが 225 回の計算になった')

---
## 課題4: 命令を表に登録する (10点)

**やること**: 「命令の名前 → 関数」の辞書 `COMMANDS` に、命令を登録する。

課題2 では「演算子 → 関数」の表を作った。今度は命令の名前で関数を引く表を作る。
下のセルには、命令にする関数が3つ書いてある。`'calc'` は登録済み。

| 名前 | 関数 | 働き |
|---|---|---|
| `'calc'` | `cmd_calc` | 2つの整数を計算する (登録済み) |
| `'shout'` | `cmd_shout` | 大文字にして `!!` を付ける |
| `'rev'` | `cmd_rev` | 文字列を逆順にする |

**手順**

1. `COMMANDS['shout'] = cmd_shout` を書く
2. `COMMANDS['rev'] = cmd_rev` を書く

**つまずきやすい点**

- `COMMANDS['shout'] = cmd_shout()` のように `()` を付けると、関数を呼んでしまう。`()` は付けない (関数そのものを入れる)

### 例題

次のセルを実行して、動きを見る。

名前と関数を辞書に入れて、名前で引いて呼ぶ。

In [ ]:
# 例題
def on_click(x, y):
    return f'clicked {x},{y}'


HANDLERS = {}
HANDLERS['click'] = on_click

print(HANDLERS['click']('1', '2'))   # clicked 1,2

In [ ]:
def cmd_calc(op, a, b):
    """2つの整数を演算する"""
    return calc(op, int(a), int(b))


def cmd_shout(text):
    """大文字にして !! を付ける"""
    return text.upper() + '!!'


def cmd_rev(text):
    """文字列を逆順にする"""
    return text[::-1]


COMMANDS = {}
COMMANDS['calc'] = cmd_calc
COMMANDS['shout'] = cmd_shout
COMMANDS['rev'] = cmd_rev

print(COMMANDS)

In [ ]:
# [CHECK] q4
need('COMMANDS', kind=dict)

assert COMMANDS.get('calc') is cmd_calc
assert COMMANDS.get('shout') is cmd_shout, "COMMANDS['shout'] = cmd_shout を書くこと"
assert COMMANDS.get('rev') is cmd_rev, "COMMANDS['rev'] = cmd_rev を書くこと"
assert COMMANDS['shout']('hi') == 'HI!!'
print('OK')

### 課題4-b: 文字列から命令を実行する (15点)

**やること**: `run(line)` を書く。`line` は `'shout hello'` のような1行の命令。
先頭の語で `COMMANDS` から関数を引き、残りの語を引数にして呼ぶ。

**入出力の例**

| 呼び方 | 結果 |
|---|---|
| `run('shout hello')` | `'HELLO!!'` |
| `run('calc * 6 7')` | `42` |
| `run('rev abc')` | `'cba'` |
| `run('unknown x')` | `ValueError` |

**手順**

1. `name, *args = line.split()` で、先頭の語 `name` と、残りの語のリスト `args` に分ける
2. `name` が `COMMANDS` に無ければ `raise ValueError(name)`
3. `COMMANDS[name](*args)` を返す

**ヒント**: `*args` は「残り全部をリストで受け取る」、`func(*args)` は「リストの中身を1つずつ引数として渡す」という意味。
引数はすべて**文字列**で渡ってくる (`'6'` と `'7'`)。数への変換は `cmd_calc` の中でしている。

**つまずきやすい点**

- `COMMANDS[name](args)` と `*` を忘れると、リストが1つの引数として渡り、エラーになる
- 結果を `print` だけして `return` していない → `None` になり NG

### 例題

次のセルを実行して、動きを見る。

文字列を、先頭と残りに分ける。

In [ ]:
# 例題
name, *args = 'shout hello world'.split()
print(name, args)    # shout ['hello', 'world']

In [ ]:
def run(line):
    name, *args = line.split()
    if name not in COMMANDS:
        raise ValueError(name)
    return COMMANDS[name](*args)


for line in ['calc * 6 7', 'shout hello', 'rev abc']:
    print(f'{line:16} -> {run(line)}')

In [ ]:
# [CHECK] q4b
need('COMMANDS', kind=dict)
need('run', arity=1)

assert run('calc * 6 7') == 42
assert run('calc ** 2 10') == 1024
assert run('shout hello') == 'HELLO!!'
assert run('rev abc') == 'cba'

try:
    run('unknown x')
except ValueError:
    pass
else:
    raise AssertionError('知らない命令は ValueError を出すこと')

# 実行中に命令を増やせること
def cmd_twice(text):
    """2回くり返す"""
    return text * 2

COMMANDS['twice'] = cmd_twice
assert run('twice ab') == 'abab'
print('OK')

---
# 発展課題

余力があれば。


## 発展1: 再帰を眺める (5点)

**やること**: `tarai_t(x, y, z)` を書く。`tarai` と同じ値を返しながら、
呼ばれるたびに引数と返り値を表示する。呼び出しが深くなるほど、字下げを増やす。

```
tarai_t(3, 2, 1)
  tarai_t(2, 2, 1)
  -> 2
  ...
-> 3
```

**手順**

1. `'  ' * DEPTH` (深さ×空白2個) で字下げして、`tarai_t(x, y, z)` を表示する
2. `DEPTH += 1` で、深さを1つ増やす
3. `tarai` と同じ式で `result` を計算する (式の中で呼ぶのは `tarai_t`)
4. `DEPTH -= 1` で深さを戻し、同じ字下げで `-> 返り値` を表示する
5. `result` を返す (**返り値は変えないこと**)

**ヒント**: 下の例題 `fib_t` は、手順1〜3 と同じことをしている。

**つまずきやすい点**

- `呼ぶたびに深さがずれている` と出る → 手順4 の `DEPTH -= 1` が抜けている

### 例題

次のセルを実行して、動きを見る。

`fib` で、呼ばれたときだけ表示する例。

In [ ]:
# 例題
FIB_DEPTH = 0


def fib_t(n):
    global FIB_DEPTH
    print('  ' * FIB_DEPTH + f'fib_t({n})')
    FIB_DEPTH += 1
    if n <= 1:
        result = n
    else:
        result = fib_t(n - 1) + fib_t(n - 2)
    FIB_DEPTH -= 1
    return result


fib_t(3)

課題は、これに「返るときの `-> 返り値`」の表示を足したもの。

In [ ]:
DEPTH = 0


def tarai_t(x, y, z):
    global DEPTH
    print('  ' * DEPTH + f'tarai_t({x}, {y}, {z})')
    DEPTH += 1
    if x <= y:
        result = y
    else:
        result = tarai_t(tarai_t(x-1, y, z), tarai_t(y-1, z, x), tarai_t(z-1, x, y))
    DEPTH -= 1
    print('  ' * DEPTH + f'-> {result}')
    return result


print('結果:', tarai_t(3, 2, 1))

In [ ]:
# [CHECK] b1
need('tarai_t', arity=3)

import io, contextlib
DEPTH = 0
buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    result = tarai_t(3, 2, 1)
out = buf.getvalue()
assert result == 3, '返り値を壊さないこと'
assert out.count('tarai_t') >= 5, '呼び出しが表示されていない'
assert '->' in out, '返り値が表示されていない'
assert any(l.startswith('  ') and not l.startswith('    ')
           for l in out.splitlines()), '深さ1 の行が字下げされていない'
assert any(l.startswith('    ') for l in out.splitlines()), '深さ2 以上の字下げが無い'

# 深さが元に戻ること (2回呼んでも字下げが増えない)
buf2 = io.StringIO()
with contextlib.redirect_stdout(buf2):
    tarai_t(3, 2, 1)
assert buf2.getvalue() == out, '呼ぶたびに深さがずれている'
print('OK')

## 発展2: `help` を自動生成する (5点)

**やること**: `help_text()` を書く。`COMMANDS` に登録された関数の**説明文と引数名**から、使い方の一覧を作って文字列で返す。

```
calc <op> <a> <b>       2つの整数を演算する
rev <text>              文字列を逆順にする
...
```

**手順**

1. 結果の行を入れる空のリストを作る
2. `sorted(COMMANDS)` で、命令の名前を辞書順に1つずつ取り出す
3. 名前ごとに、`名前 <引数1> <引数2> ...` のあとに説明文を付けた1行を作り、リストに足す
4. `'\n'.join(リスト)` で、改行でつないだ文字列を返す

**ヒント**: 関数の最初の行の `"""..."""` が説明文 (docstring) で、`func.__doc__` で取り出せる。
引数名の一覧は `inspect.signature(func).parameters` で取れる。

**ここが面白いところ**: 一覧を手で書いていない。
**プログラムが自分自身を調べて**説明を作っている。

### 例題

次のセルを実行して、動きを見る。

関数から、引数名と説明文を取り出す。

In [ ]:
# 例題
import inspect


def introduce(name, times):
    """挨拶する"""
    return ('やあ ' + name) * times


print(list(inspect.signature(introduce).parameters))   # ['name', 'times']
print(introduce.__doc__)                               # 挨拶する

あとは `COMMANDS` を辞書順に回して、1行ずつ組み立てる。

In [ ]:
import inspect


def help_text():
    out = []
    for n in sorted(COMMANDS):
        f = COMMANDS[n]
        a = ' '.join(f'<{p}>' for p in inspect.signature(f).parameters)
        out.append((n + ' ' + a).ljust(24) + (f.__doc__ or ''))
    return '\n'.join(out)


print(help_text())


In [ ]:
# [CHECK] b2
need('help_text', arity=0)

t = help_text()
assert isinstance(t, str), '文字列を返すこと'
for name in COMMANDS:
    assert name in t, f'{name} が一覧に無い'
assert '<op>' in t and '<text>' in t, '引数名を <> で囲んで並べること'
assert '逆順' in t, 'docstring を使うこと'
assert t.index('calc') < t.index('rev') < t.index('shout'), '辞書順に並べること'
assert len(t.splitlines()) >= len(COMMANDS), '1行につき1コマンド'
print('OK')


---
## 自己確認

`score()` で全部まとめて確かめる。何度やってもよい。
`NG` の行に、次に何をすればいいかが出る。


In [ ]:
# [SKIP] 採点の対象外
score()


---
## 提出

提出は kibaco から。`score()` と採点ページは下書き確認用。

1. 上から実行し、出力を残す
2. `ファイル → .ipynb をダウンロード`
3. 締切までに kibaco へ出す

確認セルが全部 `OK` なら提出してよい。確認セルを消しても教員の採点は別の検査で行う。
